# MoMADiff — T2M-CapBench Generation Runner

This notebook **defines and exports** the MoMADiff generation runner used by T2M-CapBench. It does not select Pilot/Main prompts, evaluate motions, or run the benchmark.

**Master benchmark responsibilities:** select prompts and target frames, set the seed, call `runner.generate(...)`, evaluate and aggregate results.

**Runner responsibilities:** prepare MoMADiff, load pretrained weights, generate one motion on request, standardise to `float32 [T,22,3]` (20 fps, +Y up, +Z forward, +X right), and return the result.

> Run the cells in order in **Google Colab**. Model installation, checkpoint download and inference require internet and a suitable Colab runtime. The optional test cells do **not** run automatically. This notebook is a standalone source for the runner; the Master Notebook currently has its own embedded runner, so uploading this file alone does not switch the Master Notebook to it.

## 1 — Export runner module

This cell writes the complete reusable Python module to `/content/momadiff_benchmark_runner.py`. There is **no `SPLIT` setting** in the runner. The master benchmark supplies `prompt`, `prompt_id`, `target_frames`, and `seed`.

In [ ]:
%%writefile /content/momadiff_benchmark_runner.py
"""
MoMADiff model-generation adapter for T2M-CapBench.

Purpose
-------
This module keeps MoMADiff-specific setup, repository preparation, model loading,
generation, and numerical standardisation OUTSIDE the main benchmark notebook.

Final benchmark flow:
    benchmark JSON
        -> MoMADiffRunner.generate(...)
        -> MoMADiff repository/model
        -> standardized XYZ motion [T, 22, 3]
        -> benchmark evaluators

Important
---------
- No GIF rendering is required here.
- The returned XYZ motion is numerically standardised for benchmark evaluation:
    +Y = Up
    +Z = Forward
    +X = Right
    units = metres
    initial root XZ = (0, 0)
    ground aligned to Y = 0
- The original normalized 263-D motion is also retained for evaluators such as
  Matching Score if needed.
"""

from __future__ import annotations

import os
import sys
import shutil
import pickle
import itertools
import subprocess
from pathlib import Path
from argparse import ArgumentParser
from typing import Optional, Dict, Any

import numpy as np


class MoMADiffRunner:
    MODEL_NAME = "MoMADiff"

    def __init__(
        self,
        repo_dir: str = "/content/MoMADiff",
        checkpoint_download_dir: str = "/content/checkpoints_downloaded",
        output_root: str = "/content/t2m_benchmark_outputs/MoMADiff",
        fps: float = 20.0,
    ):
        self.repo_dir = Path(repo_dir)
        self.checkpoint_download_dir = Path(checkpoint_download_dir)
        self.output_root = Path(output_root)
        self.fps = float(fps)

        self._loaded = False

        # Populated by load_model()
        self.torch = None
        self.F = None
        self.Categorical = None
        self.recover_from_ric = None
        self.filtering = None

        self.opt = None
        self.encdec_model = None
        self.latent_transformer = None
        self.diff_model = None
        self.diffusion = None
        self.length_estimator = None
        self.mean = None
        self.std = None

    # =========================================================
    # 1. Repository / environment preparation
    # =========================================================

    def setup_environment(
        self,
        install_dependencies: bool = True,
        download_checkpoints: bool = True,
    ) -> None:
        """
        Prepare the MoMADiff repository and pretrained checkpoints.

        This is designed to be run once per fresh Colab runtime.
        Existing repository/checkpoints are reused.
        """

        # Clone repository only when missing.
        if not self.repo_dir.exists():
            print("[MoMADiff] Cloning repository...")
            subprocess.check_call([
                "git", "clone",
                "https://github.com/zzysteve/MoMADiff.git",
                str(self.repo_dir),
            ])
        else:
            print(f"[MoMADiff] Repository already exists: {self.repo_dir}")

        # Install dependencies only when requested.
        if install_dependencies:
            print("[MoMADiff] Installing dependencies...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install",
                "git+https://github.com/openai/CLIP.git"
            ])
            subprocess.check_call([
                sys.executable, "-m", "pip", "install",
                "-r", str(self.repo_dir / "requirements.txt")
            ])

        required = self.required_checkpoint_files()

        if download_checkpoints and not all(p.exists() for p in required):
            print("[MoMADiff] Downloading pretrained checkpoints...")
            subprocess.check_call(
                [sys.executable, "prepare/hf_download.py"],
                cwd=str(self.repo_dir),
            )

            source_t2m = self.checkpoint_download_dir / "t2m"
            target_root = self.repo_dir / "checkpoints"
            target_t2m = target_root / "t2m"

            target_root.mkdir(parents=True, exist_ok=True)

            if not source_t2m.exists():
                raise FileNotFoundError(
                    f"Checkpoint download finished but {source_t2m} was not found."
                )

            if target_t2m.exists():
                shutil.rmtree(target_t2m)

            shutil.copytree(source_t2m, target_t2m)

        self.verify_checkpoints()
        print("[MoMADiff] Environment ready.")

    def required_checkpoint_files(self):
        base = self.repo_dir / "checkpoints" / "t2m"
        return [
            base / "kl_vae_ver0-stable" / "args_for_pretrained_klvae.pkl",
            base / "kl_vae_ver0-stable" / "net_last.pth",
            base / "length_estimator" / "model" / "finest.tar",
            base / "Trans-B_EMA_klv0-stable_masked-only_diff1000" / "model" / "latest_ema.tar",
        ]

    def verify_checkpoints(self) -> None:
        missing = [str(p) for p in self.required_checkpoint_files() if not p.exists()]
        if missing:
            raise FileNotFoundError(
                "Missing required MoMADiff checkpoint files:\n- "
                + "\n- ".join(missing)
            )

        print("[MoMADiff] Required checkpoints verified.")

    # =========================================================
    # 2. MoMADiff imports and model loading
    # =========================================================

    def _prepare_python_path(self):
        repo = str(self.repo_dir)
        if repo not in sys.path:
            sys.path.insert(0, repo)
        os.chdir(repo)

    def load_model(self) -> None:
        """
        Load KLVAE, MoMADiff, DDIM diffusion, and the length estimator.
        Call once per runtime after setup_environment().
        """
        if self._loaded:
            print("[MoMADiff] Model already loaded.")
            return

        self.verify_checkpoints()

        # MoMADiff uses a package named `common` (for example
        # `common.skeleton`). The T2M-CapBench evaluator also has a module
        # named `evaluation/common.py`. In a reused Colab kernel the latter
        # may already be present in sys.modules, which makes
        # `from common.skeleton import Skeleton` fail with:
        #     ModuleNotFoundError: 'common' is not a package
        #
        # Remove every stale common.* entry while loading MoMADiff and force
        # the MoMADiff repository to the front of sys.path. The imported
        # MoMADiff modules retain references to their own common package.
        stale_common_modules = {
            name: module
            for name, module in list(sys.modules.items())
            if name == "common" or name.startswith("common.")
        }
        for name in list(stale_common_modules):
            sys.modules.pop(name, None)

        old_cwd = os.getcwd()
        self._prepare_python_path()

        import warnings
        import torch
        import torch.nn.functional as F
        from torch.distributions.categorical import Categorical
        from torch_ema import ExponentialMovingAverage

        from models.klvae.autoencoder import AutoencoderKL as KLVAE
        from models.mask_transformer.latent_transformer import MaskLatentTransformer
        from models.length_est import LengthEstimator

        from options.train_option import (
            add_train_mode_args,
            load_config_file_and_parse,
        )

        from utils.diffusion import (
            create_model_and_diffusion,
            create_gaussian_diffusion_ddim,
        )

        from utils.fixseed import fixseed
        from utils.motion_process import recover_from_ric
        from utils.demo_util import filtering

        warnings.filterwarnings(
            "ignore",
            category=UserWarning,
            module="matplotlib.animation",
        )

        clip_version = "ViT-B/32"

        # Preserve notebook/kernel arguments while MoMADiff parses its own config.
        old_argv = sys.argv[:]
        try:
            sys.argv = [
                "benchmark",
                "--name", "Trans-B_EMA_klv0-stable_masked-only_diff1000",
                "--config", "configs/t2m.yaml",
                "--use_ema", "true",
                "--diffusion_steps", "1000",
                "--loss_strategy", "masked",
                "--ddim_steps", "100",
                "--trans_infer_timesteps", "9",
                "--guidance_param", "3",
            ]

            parser = ArgumentParser()
            parser = add_train_mode_args(parser)
            opt = load_config_file_and_parse(parser)
        finally:
            sys.argv = old_argv

        fixseed(opt.seed)

        opt.device = torch.device(
            "cpu" if opt.device == -1 else "cuda:" + str(opt.device)
        )

        # Same inference settings used in the permanent Pilot notebook.
        opt.time_steps = 7
        opt.cond_scale = 3

        # ---------- KLVAE ----------
        kl_args_path = Path(opt.checkpoints_dir) / opt.dataset_name / opt.kl_name / \
            "args_for_pretrained_klvae.pkl"

        with open(kl_args_path, "rb") as f:
            kl_args = pickle.load(f)

        encdec_model = KLVAE(
            kl_args,
            kl_args.output_emb_width,
            kl_args.down_t,
            kl_args.stride_t,
            kl_args.width,
            kl_args.depth,
            kl_args.dilation_growth_rate,
        )

        kl_args.klvae_pth = str(
            Path(opt.checkpoints_dir)
            / opt.dataset_name
            / opt.kl_name
            / "net_last.pth"
        )

        kl_ckpt = torch.load(kl_args.klvae_pth, map_location="cpu")
        encdec_model.load_state_dict(kl_ckpt["net"], strict=True)
        encdec_model = encdec_model.to(opt.device)
        encdec_model.eval()

        # ---------- MoMADiff ----------
        cond_mode = "text" if not opt.unconstrained else "uncond"

        diff_model, diffusion = create_model_and_diffusion(opt)

        latent_transformer = MaskLatentTransformer(
            code_dim=opt.code_dim,
            cond_mode=cond_mode,
            latent_dim=opt.latent_dim,
            ff_size=opt.ff_size,
            num_layers=opt.n_layers,
            num_heads=opt.n_heads,
            dropout=opt.dropout,
            clip_dim=512,
            cond_drop_prob=opt.cond_drop_prob,
            opt=opt,
            clip_version=clip_version,
            use_ema=opt.use_ema,
        )

        checkpoint_file = "latest_ema.tar"
        ckpt_path = (
            Path(opt.checkpoints_dir)
            / opt.dataset_name
            / opt.name
            / "model"
            / checkpoint_file
        )

        ckpt = torch.load(str(ckpt_path), map_location=opt.device)

        ema = ExponentialMovingAverage(
            itertools.chain(
                latent_transformer.parameters(),
                diff_model.parameters(),
            ),
            decay=opt.ema_decay,
        )
        ema.load_state_dict(ckpt)
        ema.copy_to(
            itertools.chain(
                latent_transformer.parameters(),
                diff_model.parameters(),
            )
        )

        latent_transformer = latent_transformer.to(opt.device)
        diff_model = diff_model.to(opt.device)
        latent_transformer.eval()
        diff_model.eval()

        if opt.ddim_steps != -1:
            diffusion = create_gaussian_diffusion_ddim(
                opt,
                opt.ddim_steps,
            )

        # ---------- Length estimator ----------
        length_estimator = LengthEstimator(512, 50)

        length_ckpt_path = (
            Path(opt.checkpoints_dir)
            / opt.dataset_name
            / "length_estimator"
            / "model"
            / "finest.tar"
        )

        length_ckpt = torch.load(
            str(length_ckpt_path),
            map_location=opt.device,
        )
        length_estimator.load_state_dict(length_ckpt["estimator"])
        length_estimator = length_estimator.to(opt.device)
        length_estimator.eval()

        # ---------- HumanML3D statistics ----------
        opt.nb_joints = 21 if opt.dataset_name == "kit" else 22

        mean = np.load(Path(opt.meta_dir) / "mean.npy")
        std = np.load(Path(opt.meta_dir) / "std.npy")

        # Keep only what the runner needs after loading.
        self.torch = torch
        self.F = F
        self.Categorical = Categorical
        self.recover_from_ric = recover_from_ric
        self.filtering = filtering

        self.opt = opt
        self.encdec_model = encdec_model
        self.latent_transformer = latent_transformer
        self.diff_model = diff_model
        self.diffusion = diffusion
        self.length_estimator = length_estimator
        self.mean = mean
        self.std = std

        self._loaded = True

        # Do not leave the process in /content/MoMADiff after model loading.
        # All runner paths are absolute, and restoring the caller cwd makes
        # the later MotionHiFlow / Light-T2M / evaluation stages deterministic.
        os.chdir(old_cwd)

        # If a stale evaluator `common` existed before this load, do NOT put it
        # back here: MoMADiff may import its common package lazily during
        # generation. The evaluation stage explicitly clears common.* before
        # importing evaluation/common.py.

        print("[MoMADiff] Model loaded.")
        print("[MoMADiff] Device:", opt.device)
        print("[MoMADiff] Dataset:", opt.dataset_name)

    # =========================================================
    # 3. Benchmark numerical standardisation
    # =========================================================

    @staticmethod
    def standardize_xyz(motion_xyz: np.ndarray) -> np.ndarray:
        """
        Convert recovered MoMADiff XYZ into the benchmark numerical convention.

        Convention:
          shape          : [T, 22, 3]
          coordinates    : global XYZ
          +Y             : up
          +Z             : forward
          +X             : right
          units          : metres
          initial root XZ: (0, 0)
          ground         : Y = 0

        The X mirror is applied exactly once to the returned numerical motion.
        The benchmark notebook must not mirror these positions again.
        """
        motion = np.asarray(motion_xyz, dtype=np.float32).copy()

        if motion.ndim != 3 or motion.shape[1:] != (22, 3):
            raise ValueError(
                f"Expected MoMADiff XYZ shape [T,22,3], got {motion.shape}"
            )

        if not np.isfinite(motion).all():
            raise ValueError("Motion contains NaN or infinite values.")

        # Ground alignment.
        motion[:, :, 1] -= motion[:, :, 1].min()

        # Initial root XZ -> origin.
        motion[:, :, 0] -= motion[0, 0, 0]
        motion[:, :, 2] -= motion[0, 0, 2]

        # Benchmark convention: +X = Right, -X = Left.
        motion[:, :, 0] *= -1.0

        return motion

    def _inv_transform(self, data):
        return data * self.std + self.mean

    # =========================================================
    # 4. Per-prompt generation (split-agnostic)
    # =========================================================

    def generate(
        self,
        prompt: str,
        prompt_id: str,
        capability: str = "",
        difficulty: str = "",
        target_frames: Optional[int] = None,
        seed: Optional[int] = None,
        save_outputs: bool = True,
    ) -> Dict[str, Any]:
        """
        Generate exactly one motion for the supplied prompt.

        The caller owns split selection, prompt iteration, target lengths,
        evaluation, and result aggregation. This runner is split-agnostic.

        Parameters
        ----------
        target_frames:
            Requested motion length in frames, supplied by the benchmark.
            If None, MoMADiff estimates its own length. Generation uses
            4-frame tokens; the runner rounds to a supported frame count.

        Returns
        -------
        dict with:
          positions       standardized [T,22,3] XYZ used by benchmark
          eval_motion     original normalized 263-D representation
          metadata        common model/run metadata
          paths           saved numeric files, when save_outputs=True

        No GIF is generated.
        """
        if not self._loaded:
            raise RuntimeError(
                "MoMADiff is not loaded. Run setup_environment() and load_model() first."
            )

        torch = self.torch
        F = self.F
        Categorical = self.Categorical
        opt = self.opt

        if seed is not None:
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

        # Benchmark共通インターフェースでは、生成長をフレーム数で受け取る。
        # target_frames=None の場合は、MoMADiff自身のLength Estimatorを使用する。
        # MoMADiff内部では 1 token = 4 frames なので、指定されたフレーム数を
        # Runner内部でtoken長へ変換する。
        if target_frames is None:
            with torch.no_grad():
                text_embedding = self.latent_transformer.encode_text(prompt)
                pred_dis = self.length_estimator(text_embedding)
                probs = F.softmax(pred_dis, dim=-1)
                token_lens = Categorical(probs).sample()[:1]
        else:
            if not isinstance(target_frames, (int, np.integer)):
                raise TypeError(
                    f"target_frames must be an integer or None, got {type(target_frames).__name__}."
                )

            if target_frames <= 0:
                raise ValueError(
                    f"target_frames must be greater than 0, got {target_frames}. "
                    "Use None to let MoMADiff estimate the motion length."
                )

            # MoMADiff can represent lengths in 4-frame units.
            token_length = max(1, int(round(target_frames / 4)))

            token_lens = torch.tensor(
                [token_length],
                device=opt.device,
                dtype=torch.long,
            )

        m_length = int((token_lens * 4).item())

        with torch.no_grad():
            pred_latent, _gen_steps = self.latent_transformer.generate(
                [prompt],
                token_lens,
                opt.time_steps,
                opt.cond_scale,
                diffusion=self.diffusion,
                diff_model=self.diff_model,
                output_inference_step=True,
                noise_schedule=None,
            )

            pred_motions_eval = self.encdec_model.decode(pred_latent)

        # Exact normalized 263-D generation.
        eval_motion = (
            pred_motions_eval[0, :m_length]
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        # Same exact generation -> recovered XYZ.
        visual_data = self._inv_transform(
            pred_motions_eval.detach().cpu().numpy()
        )
        visual_data = visual_data[:, :m_length]

        raw_xyz = self.recover_from_ric(
            torch.from_numpy(visual_data).float(),
            22,
        ).numpy()[0]

        raw_xyz = self.filtering(
            raw_xyz,
            m_length,
            window_size=15,
            poly_order=3,
        ).astype(np.float32)

        positions = self.standardize_xyz(raw_xyz)

        # Check the canonical benchmark contract before returning the result.
        # This does not modify the motion or apply a second axis conversion.
        if positions.dtype != np.float32:
            raise ValueError(f"Canonical motion must be float32, got {positions.dtype}")
        if positions.ndim != 3 or positions.shape[1:] != (22, 3):
            raise ValueError(f"Invalid canonical motion shape: {positions.shape}")
        if positions.shape[0] != m_length:
            raise ValueError(
                f"Generated {positions.shape[0]} frames, expected {m_length}"
            )
        if not np.isfinite(positions).all():
            raise ValueError("Canonical motion contains NaN or infinity")
        if not np.allclose(positions[0, 0, [0, 2]], 0, atol=1e-5):
            raise ValueError("Initial root XZ must be (0, 0)")
        if not np.isclose(float(positions[:, :, 1].min()), 0, atol=1e-5):
            raise ValueError("Ground must be aligned to Y=0")

        paths = {}

        if save_outputs:
            safe_prompt_id = "".join(
                c if c.isalnum() or c in "-_" else "_"
                for c in prompt_id
            )

            run_dir = self.output_root / safe_prompt_id
            run_dir.mkdir(parents=True, exist_ok=True)

            eval_path = run_dir / "eval_motion.npy"
            raw_xyz_path = run_dir / "motion_xyz_raw.npy"
            xyz_path = run_dir / "motion_xyz.npy"

            np.save(eval_path, eval_motion)
            np.save(raw_xyz_path, raw_xyz)
            np.save(xyz_path, positions)

            paths = {
                "eval_motion": str(eval_path),
                "motion_xyz_raw": str(raw_xyz_path),
                "motion_xyz": str(xyz_path),
            }

        metadata = {
            "model_name": self.MODEL_NAME,
            "prompt_id": prompt_id,
            "prompt_text": prompt,
            "capability": capability,
            "difficulty": difficulty,
            "target_frames": target_frames,
            "frames": int(positions.shape[0]),
            "length_source": "model_estimated" if target_frames is None else "benchmark_target",
            "fps": self.fps,
            "shape": list(positions.shape),
            "coordinate_system": "global_xyz",
            "up_axis": "+Y",
            "forward_axis": "+Z",
            "right_axis": "+X",
            "units": "metres",
            "joint_count": 22,
            "visualisation_generated": False,
            "seed": seed,
        }

        return {
            "positions": positions,
            "eval_motion": eval_motion,
            "raw_positions": raw_xyz,
            "metadata": metadata,
            "paths": paths,
        }


# =============================================================
# Optional convenience function for the benchmark notebook
# =============================================================

_DEFAULT_RUNNER: Optional[MoMADiffRunner] = None


def get_momadiff_runner(
    setup: bool = False,
    install_dependencies: bool = True,
    download_checkpoints: bool = True,
) -> MoMADiffRunner:
    """
    Return one reusable MoMADiffRunner instance.

    Example in the master benchmark notebook:
        runner = get_momadiff_runner(setup=True)
        result = runner.generate(
            prompt="A person walks forward.",
            prompt_id="C1-01",
            target_frames=100
        )
    """
    global _DEFAULT_RUNNER

    if _DEFAULT_RUNNER is None:
        _DEFAULT_RUNNER = MoMADiffRunner()

    if setup:
        _DEFAULT_RUNNER.setup_environment(
            install_dependencies=install_dependencies,
            download_checkpoints=download_checkpoints,
        )
        _DEFAULT_RUNNER.load_model()

    return _DEFAULT_RUNNER


## 2 — Check that the runner can be imported

This checks the Python module interface only. It does not download weights or run generation.

In [ ]:
import sys
import importlib

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import momadiff_benchmark_runner as moma
importlib.reload(moma)

runner = moma.get_momadiff_runner(setup=False)
assert hasattr(runner, "generate")
assert hasattr(runner, "setup_environment")
assert hasattr(runner, "load_model")
print("MoMADiff runner module imported successfully.")
print("Ready for setup and benchmark calls; no model has been loaded yet.")

## 3 — Optional: prepare MoMADiff in Colab

Run this **only if you want to test inference here**. It clones the upstream repository, installs dependencies, downloads checkpoints and loads the model. The master benchmark normally performs this setup itself.

In [ ]:
# Optional setup — uncomment the following line to run it.
# runner = moma.get_momadiff_runner(
#     setup=True,
#     install_dependencies=True,
#     download_checkpoints=True,
# )

## 4 — Optional: generate one motion (smoke test)

Run Section 3 first. These are **example inputs only**, not a Pilot/Main selector. The generated motion is returned in `result["positions"]`.

In [ ]:
# Optional smoke test — uncomment to generate a single motion.
# result = runner.generate(
#     prompt="A person walks forward.",
#     prompt_id="SMOKE-01",
#     capability="basic_action",
#     difficulty="Easy",
#     target_frames=100,
#     seed=1,
#     save_outputs=False,
# )
# positions = result["positions"]
# assert positions.shape == (100, 22, 3), positions.shape
# print("Generated canonical motion:", positions.shape, positions.dtype)

## 5 — Master Notebook integration

The master benchmark should call the same module interface:

```python
import momadiff_benchmark_runner as moma
runner = moma.get_momadiff_runner(setup=True)
result = runner.generate(
    prompt=prompt,
    prompt_id=prompt_id,
    capability=capability,
    difficulty=difficulty,
    target_frames=comparison_length,
    seed=SEED,
    save_outputs=False,
)
positions = result["positions"]
```

**Important:** `%%writefile` writes to the current Colab runtime, not automatically to GitHub. For the Master Notebook to use this version, export the generated `/content/momadiff_benchmark_runner.py` or adjust its embedded writefile cell. Keep the master and standalone copies synchronised. Do not run a second embedded `%%writefile` after importing if it would overwrite this module.